# M1.5 · Memoria 📒

**Lección oficial:** *Memory* · **Notebook original:** `module-1/1.3_memory.ipynb`

## 🎯 Qué vas a aprender
- Por qué un agente **olvida** todo entre preguntas.
- Cómo darle memoria con un **checkpointer** (`InMemorySaver`).
- Qué es un **thread** (`thread_id`): conversaciones separadas.

## 🧸 Explicado sencillo
- El **checkpointer** es una **libreta** 📒 donde el agente anota la conversación.
- El **`thread_id`** es el **número de página**.

Si vuelves a la misma página, el agente lee lo que anotó. Si abres una página nueva, empieza de cero.

🧠 **Mnemotecnia:** *checkpointer = libreta, thread_id = página.*

## 🛠️ Paso 0 · Preparar el notebook

Esta celda hace tres cosas por ti:
1. Encuentra la carpeta principal del repo (donde vive `local_model.py`).
2. Lee tu archivo `.env` (tus llaves y la dirección del servidor) **sin mostrarlas**.
3. Revisa el **semáforo** del servidor Ollama, que es **compartido**:
   - 🟢 libre → adelante.
   - 🟡 hay un modelo cargado → si fuiste tú hace un rato, sigue; si no, espera.
   - 🔴 sin conexión → revisa `OLLAMA_BASE_URL` en tu `.env`.

> Corre las celdas **en orden, de arriba hacia abajo** (Shift + Enter). El número `[n]` a la izquierda te dice en qué orden se ejecutaron.

In [ ]:
# 🟢 ADAPTADO LOCAL: preparar entorno (ver ayuda.py)
from ayuda import preparar, ver, semaforo
preparar()

from local_model import get_model   # se importa DESPUÉS de cargar el .env

## 1 · Sin memoria 😶
Le decimos nuestro nombre y color favorito, y luego le preguntamos el color.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre
# agent = create_agent("gpt-5-nano")

# 🟢 ADAPTADO LOCAL
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=get_model())

response = agent.invoke({"messages": [HumanMessage(content="Hello, my name is Seán and my favourite colour is green")]})
ver(response)

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="What's my favourite colour?")]})
ver(response)

🤔 **¿Por qué no lo sabe?** Cada `invoke` es una conversación **nueva**: solo ve el mensaje que le mandas en ese momento. Nadie anotó nada.

## 2 · Con memoria 📒
Agregamos dos cosas:
1. `checkpointer=InMemorySaver()` → la libreta (vive en la RAM: si reinicias el kernel, se borra).
2. `config` con `thread_id` → la página.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model=get_model(), checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "1"}}      # página 1

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, my name is Seán and my favourite colour is green")]},
    config,
)
ver(response)

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="What's my favourite colour?")]}, config)
ver(response)

🔍 **Fíjate:** ahora `ver()` muestra **toda la conversación**, porque el agente leyó la página 1 de la libreta antes de responder.

## 3 · Otra página = otra conversación 📄
¿Qué pasa si preguntamos lo mismo en la página `"2"`?

In [ ]:
otra_pagina = {"configurable": {"thread_id": "2"}}
response = agent.invoke({"messages": [HumanMessage(content="What's my favourite colour?")]}, otra_pagina)
print(response["messages"][-1].content)

<details><summary>¿Qué debió pasar?</summary>

No lo sabe: la página 2 está en blanco. Así funcionan los chats con varias conversaciones: cada una es un `thread_id` distinto.
</details>

## ✍️ Tu turno
1. En el thread `"1"` cuéntale 3 cosas tuyas (por ejemplo, tu comida, tu música y tu animal favorito). Luego pregúntale una de ellas.
2. Crea un thread `"tareas"` y úsalo como lista: *"Recuérdame estudiar tools"*, y después *"¿Qué te pedí recordar?"*.
3. Reinicia el kernel (Kernel → Restart), corre solo lo necesario y pregunta de nuevo en el thread `"1"`. ¿Se acuerda? ¿Por qué?

## ✅ Mini quiz
1. ¿Qué hace el checkpointer?
2. ¿Qué es el `thread_id`?
3. ¿Por qué `InMemorySaver` olvida todo si reinicias el kernel?

<details><summary>Respuestas</summary>

1. Guarda la conversación (los mensajes) entre llamadas.
2. El identificador de la conversación: la "página" de la libreta.
3. Porque guarda en la memoria RAM; para que dure hace falta una base de datos (lo verás en cursos siguientes).
</details>

## 🧠 Resumen en una línea
**Sin checkpointer, el agente es un pez 🐟 (olvida todo). Checkpointer = libreta, thread_id = página.**